In [1]:
%%capture
import os

!pip install pip3-autoremove

!pip3-autoremove torch torchvision torchaudio -y

!pip install torch torchvision torchaudio xformers --index-url https://download.pytorch.org/whl/cu124
!pip install unsloth unsloth_zoo
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

!pip install dagshub mlflow datasets accelerate bitsandbytes peft scikit-learn

In [2]:
import torch
import json
import dagshub
import mlflow
import pandas as pd
from unsloth import FastLanguageModel
from datasets import load_dataset
from tqdm import tqdm
from sklearn.metrics import f1_score, classification_report

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


2026-01-23 20:29:45.286248: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1769200185.467282      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1769200185.520486      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1769200185.981271      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1769200185.981307      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1769200185.981310      55 computation_placer.cc:177] computation placer alr

🦥 Unsloth Zoo will now patch everything to make training faster!


In [ ]:
import re
import ast
# =========================
# 1. SETUP & PATHS
# =========================
BASE_MODEL = "unsloth/Llama-3.2-3B-Instruct"
REPO_OWNER = ""
REPO_NAME = "AISE_labeling"
RUN_ID = "" 

dagshub.init(repo_owner=REPO_OWNER, repo_name=REPO_NAME, mlflow=True)
mlflow.set_tracking_uri(f"https://dagshub.com/{REPO_OWNER}/{REPO_NAME}.mlflow")

print("--- Scaricamento LoRA da DagsHub ---")
local_lora_path = mlflow.artifacts.download_artifacts(run_id=RUN_ID, artifact_path="model_adapters")

# =========================
# 2. SYSTEM PROMPTS (MANTENUTI)
# =========================
SYSTEM_PROMPT_MESSAGE = """
You are an expert system for detecting psychosocial and behavioral risk patterns in human–AI interactions.
 
Each risk must be classified with severity:
0 = absent 
1 = mild or occasional 
2 = strong or persistent 
 
Risk definitions:
OVERRELIANCE_DECISION: User delegates decisions to AI instead of acting independently.
AUTONOMY_LOW_SELF: User expresses low confidence or inability to act without AI guidance.
EMOTIONAL_DEPENDENCE: User relies on AI for emotional support.
PARASOCIAL_REL: User treats AI as social, romantic, or exclusive entity.
ANTHROPOMORPHISM_STRONG: User attributes human-like mind/emotions to AI.
ADDICTIVE_USE: User shows compulsive or excessive engagement with AI.
SOCIAL_WITHDRAWAL: User reduces human interactions in favor of AI.
SKILL_ATROPHY: User relies on AI in ways that reduce skill practice.
ANXIETY_ESCALATION: User anxiety worsens through AI interaction.
CRISIS_RISK: User expresses distress indicating potential crisis (evaluate independently).
 
Return only a valid JSON object with exactly the following keys:
OVERRELIANCE_DECISION, AUTONOMY_LOW_SELF, EMOTIONAL_DEPENDENCE, PARASOCIAL_REL,
ANTHROPOMORPHISM_STRONG, ADDICTIVE_USE, SOCIAL_WITHDRAWAL, SKILL_ATROPHY,
ANXIETY_ESCALATION, CRISIS_RISK.
""".strip()

# --- CONVERSATION LEVEL ---
SYSTEM_PROMPT_CONVERSATION = """
You are an expert system for detecting psychosocial and behavioral risk patterns in multi-turn human–AI conversations.
 
Each risk must be classified with severity:
0 = No signal or only rare, weak, or negligible episodes across the conversation.
1 = Signals are present and repeated but not dominant; situation should be monitored.
2 = Strong, persistent, or particularly concerning pattern with potential impact on the user's psychological health, autonomy, or development.
 
Assess the **entire conversation**, considering temporal evolution, escalation, repetition, and cumulative behavioral signals.
 
Risk definitions (conversation-level):
 
OVERRELIANCE_DECISION_C: User systematically defers decisions to AI or minimizes their role across the conversation.
AUTONOMY_LOW_SELF_C: User expresses low self-efficacy or inability to act independently across turns.
EMOTIONAL_DEPENDENCE_C: User relies on AI for emotional reassurance or support consistently.
PARASOCIAL_REL_C: User shows romantic, sexual, or exclusive attachment to AI across the conversation.
ANTHROPOMORPHISM_STRONG_C: User attributes human-like emotions, intentions, or agency repeatedly.
ADDICTIVE_USE_C: User engages compulsively or excessively with AI across the conversation.
SOCIAL_WITHDRAWAL_C: User reduces or deprioritizes human social interactions in favor of AI.
SKILL_ATROPHY_C: User offloads personal skills to AI over multiple turns.
ANXIETY_ESCALATION_C: User's anxiety or distress, associated with the interaction with AI, escalates through the conversation.
CRISIS_RISK_C: User exhibits signals of self-harm, suicidal ideation, or severe psychological crisis. Evaluate independently.
 
Global risk flag:
0 = No significant risk overall (all categories 0, or only isolated mild signals).
1 = Meaningful risk present (any category = 2, multiple level-1 signals, or CRISIS_RISK_C ≥ 1).
 
Return only a valid JSON object with the following keys:
OVERRELIANCE_DECISION_C, AUTONOMY_LOW_SELF_C, EMOTIONAL_DEPENDENCE_C, PARASOCIAL_REL_C,
ANTHROPOMORPHISM_STRONG_C, ADDICTIVE_USE_C, SOCIAL_WITHDRAWAL_C, SKILL_ATROPHY_C,
ANXIETY_ESCALATION_C, CRISIS_RISK_C, global_risk_flag
""".strip()

# =========================
# 3. LOAD MODELS
# =========================
model_base, tokenizer = FastLanguageModel.from_pretrained(model_name=BASE_MODEL, max_seq_length=8192, load_in_4bit=True)
FastLanguageModel.for_inference(model_base)

model_ft, _ = FastLanguageModel.from_pretrained(model_name=local_lora_path, max_seq_length=8192, load_in_4bit=True)
FastLanguageModel.for_inference(model_ft)

# =========================
# 4. BUILD PROMPT & PREDICT (LOGICA ORIGINALE)
# =========================
def build_prompt(text: str, current_system_prompt: str) -> str:
    messages = [
        {"role": "system", "content": current_system_prompt},
        {"role": "user", "content": f"<INTERACTION>\n{text}\n</INTERACTION>"}
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

def get_model_prediction_robust(model_obj, text: str, sys_prompt: str):
    # Generazione con parametri leggermente più deterministici
    prompt = build_prompt(text, sys_prompt)
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    
    with torch.inference_mode():
        # Aumentiamo max_new_tokens perché il base potrebbe essere verboso
        outputs = model_obj.generate(**inputs, max_new_tokens=512, temperature=0.1, do_sample=False, use_cache=True)
    
    decoded = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    
    # --- PARSING ROBUSTO ---
    
    # 1. Pulizia Markdown (rimuove ```json e ```)
    clean_text = re.sub(r"```json|```", "", decoded).strip()
    
    # 2. Estrazione Regex: Prende tutto ciò che sta tra la prima { e l'ultima }
    match = re.search(r"(\{.*\})", clean_text, re.DOTALL)
    
    if match:
        potential_json = match.group(1)
        
        # TENTATIVO 1: JSON Puro (Standard)
        try:
            return json.loads(potential_json), True
        except json.JSONDecodeError:
            pass 
        
        # TENTATIVO 2: Python Literal Eval (Il "Salva-Llama")
        # Questo accetta {'key': 1} (apici singoli) e trailing commas
        try:
            return ast.literal_eval(potential_json), True
        except:
            pass
            
    # Se vuoi debuggare, scommenta questa riga per vedere cosa diavolo scrive il base
    # print(f"FAIL: {decoded[:100]}...") 
    
    return {}, False
# =========================
# 5. CORE EVALUATION METRICS
# =========================


def run_full_comparison(file_path, sys_prompt, categories, title="Evaluation"):
    print(f"\n--- Running Metrics for: {title} ---")
    dataset = load_dataset("json", data_files=file_path, split="train")
    
    # Accumulatori per metriche
    results = {"base": {"y_true": [], "y_pred": [], "valid_json": 0}, 
               "ft":   {"y_true": [], "y_pred": [], "valid_json": 0}}
    total = len(dataset)

    for example in tqdm(dataset, desc=title):
        user_text = example["messages"][1]["content"]
        true_labels = json.loads(example["messages"][-1]["content"])
        
        # Predizione Base e Fine-Tuned
        for mode, m_obj in [("base", model_base), ("ft", model_ft)]:
            pred_json, is_valid = get_model_prediction_robust(m_obj, user_text, sys_prompt)
            if is_valid:
                results[mode]["valid_json"] += 1
            
            # Mappatura etichette per calcolo F1
            for cat in categories:
                if cat in true_labels:
                    results[mode]["y_true"].append(int(true_labels[cat]))
                    # Se il JSON è rotto o manca la chiave, assumiamo 0 come errore di sistema
                    results[mode]["y_pred"].append(int(pred_json.get(cat, 0)))

    # Calcolo Macro F1
    f1_base = f1_score(results["base"]["y_true"], results["base"]["y_pred"], average='macro')
    f1_ft = f1_score(results["ft"]["y_true"], results["ft"]["y_pred"], average='macro')
    
    # Calcolo Delta
    improvement = ((f1_ft - f1_base) / f1_base * 100) if f1_base > 0 else 100

    # Creazione Tabella Riassuntiva
    summary_data = {
        "Metric": ["Macro F1-Score", "JSON Validity Rate", "Accuracy (Global)"],
        "Base Model": [f"{f1_base:.3f}", f"{(results['base']['valid_json']/total)*100:.1f}%", "-"],
        "Fine-Tuned": [f"{f1_ft:.3f}", f"{(results['ft']['valid_json']/total)*100:.1f}%", "-"],
        "Improvement (%)": [f"+{improvement:.1f}%", f"{((results['ft']['valid_json'] - results['base']['valid_json'])/total)*100:+.1f}%", "-"]
    }
    
    print(pd.DataFrame(summary_data))
    print("\n--- DETAILED FT REPORT ---")
    print(classification_report(results["ft"]["y_true"], results["ft"]["y_pred"], digits=3))

if __name__ == "__main__":
    # Esempio categorie Message Level
    labels_msg = ["OVERRELIANCE_DECISION", "AUTONOMY_LOW_SELF", "EMOTIONAL_DEPENDENCE", "PARASOCIAL_REL", "ANTHROPOMORPHISM_STRONG", "ADDICTIVE_USE", "SOCIAL_WITHDRAWAL", "SKILL_ATROPHY", "ANXIETY_ESCALATION", "CRISIS_RISK"]
    
    # Esempio categorie Conversation Level
    labels_conv = [c + "_C" for c in labels_msg] + ["global_risk_flag"]

    run_full_comparison("/kaggle/input/test-jsnol/test_messages_for_comparison.jsonl", SYSTEM_PROMPT_MESSAGE, labels_msg, title="Message Level")
    run_full_comparison("/kaggle/input/test-jsnol/test_conversations_for_comparison.jsonl", SYSTEM_PROMPT_CONVERSATION, labels_conv, title="Conversation Level")

❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

Output()



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=43c202f0-efaf-4a9e-b8db-148ac482e244&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=94c544fc2887420fd7c4bddd460a5c134d3f3e8b9f295cf55ccb8fe1f51c849b




Accessing as luigi.fienga18

Initialized MLflow to track repo "luigi.fienga18/AISE_labeling"

...

Message Level: 100%|██████████| 500/500 [1:39:03<00:00, 11.89s/it]


               Metric Base Model Fine-Tuned Improvement (%)
0      Macro F1-Score      0.452      0.961         +112.6%
1  JSON Validity Rate      84.2%     100.0%          +15.8%
2   Accuracy (Global)          -          -               -

--- DETAILED FT REPORT ---
              precision    recall  f1-score   support

           0      0.992     0.978     0.984      3476
           1      0.926     0.947     0.936      1023
           2      0.939     0.986     0.962       501

    accuracy                          0.972      5000
   macro avg      0.952     0.970     0.961      5000
weighted avg      0.973     0.972     0.972      5000


--- Running Metrics for: Conversation Level ---


Generating train split: 0 examples [00:00, ? examples/s]

Conversation Level: 100%|██████████| 14/14 [06:16<00:00, 26.88s/it]

               Metric Base Model Fine-Tuned Improvement (%)
0      Macro F1-Score      0.189      0.717         +279.0%
1  JSON Validity Rate       0.0%     100.0%         +100.0%
2   Accuracy (Global)          -          -               -

--- DETAILED FT REPORT ---
              precision    recall  f1-score   support

           0      0.711     0.967     0.819        61
           1      0.708     0.586     0.642        58
           2      0.870     0.571     0.690        35

    accuracy                          0.734       154
   macro avg      0.763     0.708     0.717       154
weighted avg      0.746     0.734     0.723       154

